In [1]:
import pandas as pd
from database import read_registro, read_resultados
import os
os.chdir("..")

In [3]:
import psycopg2

endpoint = ""
port = "5432"
database = "postgresql"
user = "postgres"
password = "a8091993"


# Connect to AWS RDS Postgres DB
conn = psycopg2.connect(
    host=endpoint,
    port=port,
    database=database,
    user=user,
    password=password
)

# Create a cursor object
cur = conn.cursor()

# Execute a query
cur.execute("SELECT version();")

# Fetch and print the result of the query
db_version = cur.fetchone()
print(db_version)

# Close the cursor and connection
cur.close()
conn.close()

OperationalError: connection to server at "postgresqldb.cfi4ko86mdw8.us-east-1.rds.amazonaws.com" (172.31.18.21), port 5432 failed: Connection timed out
	Is the server running on that host and accepting TCP/IP connections?


In [2]:
os.listdir()

['.vscode',
 'pyproject.toml',
 'uv.lock',
 '.git',
 '.devcontainer',
 'requirements.txt',
 'tasks.py',
 '.gitignore',
 'app',
 '.venv',
 '.python-version',
 '.gitattributes',
 'README.md',
 'data']

In [12]:
from app.database import read_resultados

data = read_resultados()
data.to_csv("data/resultados.csv", index=False)

/home/selawad/personas_venezolanas/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [23]:
import duckdb
import pyarrow.dataset as ds
from pyarrow.parquet import ParquetDataset as pq

con = duckdb.connect()

# Read the CSV file with duckdb, ignoring errors
con.execute("CREATE TABLE registro AS SELECT * FROM read_csv_auto('data/registro_electoral_nacional.csv', ignore_errors=True)")
con.execute("CREATE TABLE resultados AS SELECT * FROM read_csv_auto('data/resultados.csv', ignore_errors=True)")



WHERES = "AND primer_apellido LIKE 'EL AWAD' or primer_apellido LIKE 'CHEBIB' or segundo_apellido LIKE 'EL AWAD' or segundo_apellido LIKE 'CHEBIB'"

query = f"""
    SELECT cedula,primer_apellido,segundo_apellido,primer_nombre,segundo_nombre,Estado,Municipio,Parroquia 
    FROM registro
    LEFT JOIN (SELECT DISTINCT Estado, Municipio, Parroquia, codigo_viejo FROM resultados) AS r
    ON registro.cod_centro = r.codigo_viejo
    WHERE 1=1
    {WHERES}
    LIMIT 100
"""

data = con.execute(query).df()
data
# data = con.execute("SELECT * FROM registro WHERE primer_apellido LIKE 'EL AWAD' limit 100").df()




,cedula,primer_apellido,segundo_apellido,primer_nombre,segundo_nombre,Estado,Municipio,Parroquia
0,10417367,CHEBIB,DE HELOU,SANYA,None,Zulia,CE. MARACAIBO,PQ. CHIQUINQUIRA
1,8455861,CHEBIB,DE YORDI,ANAAM,None,Anzoátegui,MP. SIMON RODRIGUEZ,CM. EL TIGRE
2,10060808,EL AWAD,CHIBIB,JOSE,None,Anzoátegui,MP. SIMON RODRIGUEZ,CM. EL TIGRE
3,10060304,ELYORDI,CHEBIB,AMIR,MUNIR,Anzoátegui,MP. SOTILLO,CM. PUERTO LA CRUZ
4,10068708,EL AWAD,ASSAFIN,ADNAN,None,Anzoátegui,MP. SIMON RODRIGUEZ,CM. EL TIGRE
5,14944113,MATAR,CHEBIB,JISEL,None,Aragua,CE. GIRARDOT,PQ. ANDRES ELOY BLANCO
6,20080562,AZRAK,EL AWAD,ANTONIO,,Bolívar,CE. HERES,PQ. CATEDRAL
7,5992772,EL AWAD,CHEBIB,SAMI,,Anzoátegui,MP. SIMON RODRIGUEZ,CM. EL TIGRE
8,5992773,EL AWAD,CHEBIB,SAMIR,None,Anzoátegui,MP. SIMON RODRIGUEZ,CM. EL TIGRE
9,16850502,MATAR,CHEBIB,CHARBEL,NABIH,Aragua,CE. GIRARDOT,PQ. ANDRES ELOY BLANCO
